In [1]:
import os
import pickle
import numpy as np
from collections import defaultdict

In [2]:
def word_overlap(q, r):
    q_set = set(q.lower().split())
    r_set = set(r.lower().split())
    return len(q_set & r_set) / max(1, len(q_set | r_set))

def find_sim_ent_from_embs(question_ent, summary_ent, question_embs, ans_embeds,
                           emb_threshold=0.7, token_overlap_threshold=0.3,
                           token_overlap=True):
    best_match = {}
    q_entity_coverage = 0
    not_covered_q_ent = []

    for i in range(len(question_ent)):
        best_sim = emb_threshold
        best_a_ent = None
        for j in range(len(summary_ent)):
            similarity = np.dot(question_embs[i], ans_embeds[j]) / (
                np.linalg.norm(ans_embeds[j]) * np.linalg.norm(question_embs[i])
            )
            if token_overlap:
                matched = similarity > best_sim and word_overlap(question_ent[i], summary_ent[j]) >= token_overlap_threshold
            else:
                matched = similarity > best_sim
            if matched:
                best_sim = similarity
                best_a_ent = summary_ent[j]
        if best_a_ent:
            best_match[question_ent[i]] = best_a_ent
            q_entity_coverage += 1
        else:
            not_covered_q_ent.append(question_ent[i])

    recall = round(q_entity_coverage / len(question_ent), 2)
    return best_match, list(best_match.keys()), not_covered_q_ent, recall


In [11]:
word_overlap('965 ng/ ml', '83 mg/ ml')

0.2

In [7]:
PRECOMPUTE_DIR = "/projectnb/vkolagrp/yiliu/QA_pipeline/Result_rebuttal/ablation/precomputed"
with open(os.path.join(PRECOMPUTE_DIR, "MedQA_ents.pkl"), "rb") as f:
    all_records = pickle.load(f)

EMB_T, TOK_T = 0.7, 0.3
IDX = 3 # ← 改这里

by_model = defaultdict(list)
for r in all_records:
    if r["background_ents"] and len(r["b_embs"]) > 0 and len(r["r_embs"]) > 0:
        by_model[r["model"]].append(r)

for model, recs in by_model.items():
    rec = recs[IDX]
    match_emb, _, _, recall_emb = find_sim_ent_from_embs(
        rec["background_ents"], rec["response_ents"], rec["b_embs"], rec["r_embs"],
        emb_threshold=EMB_T, token_overlap=False
    )
    match_both, _, _, recall_both = find_sim_ent_from_embs(
        rec["background_ents"], rec["response_ents"], rec["b_embs"], rec["r_embs"],
        emb_threshold=EMB_T, token_overlap_threshold=TOK_T, token_overlap=True
    )

    print(f"{'='*60}")
    print(f"Model: {model}  (record #{IDX})")
    print(f"Background: {rec['original']['background'][:80]}...")
    print(f"  [emb only]          recall={recall_emb}  matched={len(match_emb)}")
    for k, v in match_emb.items():
        print(f"    {k!r:40s} → {v!r}")
    print(f"  [emb + tok_overlap]  recall={recall_both}  matched={len(match_both)}")
    for k, v in match_both.items():
        print(f"    {k!r:40s} → {v!r}")
    filtered = set(match_emb) - set(match_both)
    if filtered:
        print(f"  [filtered out]  (tok_overlap < {TOK_T})")
        for k in filtered:
            tok = word_overlap(k, match_emb[k])
            print(f"    {k!r:40s} → {match_emb[k]!r}  tok_overlap={tok:.3f}")
    print()


Model: 0.5B-Qwen25  (record #3)
Background: A 39-year-old woman is brought to the emergency department because of fevers, ch...
  [emb only]          recall=0.26  matched=10
    'laboratory studies'                     → 'laboratory findings'
    'temperature'                            → 'fever'
    '83 mg/ml'                               → '83 mg'
    'chills'                                 → 'chills'
    'fevers'                                 → 'fever'
    'platelet count'                         → 'platelets'
    'mucopurulent discharge'                 → 'mucopurulent discharge'
    'left lower quadrant'                    → 'left lower quadrant'
    'lower quadrant pain'                    → 'lower quadrant pain'
    'cervical os'                            → 'cervical os'
  [emb + tok_overlap]  recall=0.21  matched=8
    'laboratory studies'                     → 'laboratory findings'
    '83 mg/ml'                               → '83 mg'
    'chills'                        